<a href="https://colab.research.google.com/github/simoesya/ProjetoFinal/blob/main/Projeto_RAG_TriviaQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

for pasta in ["data", "results", "checkpoints"]:
    (PROJECT_DIR / pasta).mkdir(exist_ok=True)

print("Pasta do projeto criada:", PROJECT_DIR)

Pasta do projeto criada: /content/drive/MyDrive/Projeto_RAG_TriviaQA


In [ ]:
import torch

print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memória GPU:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

GPU disponível: True
GPU: Tesla T4
Memória GPU: 14.56 GB


In [ ]:
import urllib.request

url = "https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz"
arquivo = PROJECT_DIR / "data" / "triviaqa-rc.tar.gz"

if arquivo.exists():
    print("O ficheiro já existe:", arquivo)
else:
    print("A descarregar o TriviaQA. Aguarda...")
    temporario = arquivo.with_suffix(".part")
    urllib.request.urlretrieve(url, temporario)
    temporario.rename(arquivo)
    print("Download concluído!")

print(f"Tamanho: {arquivo.stat().st_size / 1024**3:.2f} GB")

A descarregar o TriviaQA. Aguarda...
Download concluído!
Tamanho: 2.48 GB


In [ ]:
import tarfile
from pathlib import PurePosixPath

DATA_DIR = PROJECT_DIR / "data" / "triviaqa"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("A extrair os ficheiros da Wikipédia. Aguarda...")

with tarfile.open(arquivo, "r:gz") as tar:
    for membro in tar:
        partes = PurePosixPath(membro.name).parts

        perguntas = membro.name.endswith((
            "qa/wikipedia-train.json",
            "qa/verified-wikipedia-dev.json"
        ))
        documento = "evidence" in partes and "wikipedia" in partes

        if membro.isfile() and (perguntas or documento):
            tar.extract(membro, path=DATA_DIR, filter="data")

print("Extração concluída!")

for ficheiro in DATA_DIR.rglob("*wikipedia*.json"):
    print(ficheiro.relative_to(DATA_DIR))

A extrair os ficheiros da Wikipédia. Aguarda...
Extração concluída!
qa/verified-wikipedia-dev.json
qa/wikipedia-train.json


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa"
)

with open(DATA_DIR / "qa/wikipedia-train.json", encoding="utf-8") as f:
    perguntas = json.load(f)["Data"]

with open(DATA_DIR / "qa/verified-wikipedia-dev.json", encoding="utf-8") as f:
    teste = json.load(f)["Data"]

# As primeiras 7900 perguntas são para validação.
validacao = perguntas[:7900]
treino = perguntas[7900:]

assert len(validacao) == 7900

print("Perguntas de treino:", len(treino))
print("Perguntas de validação:", len(validacao))
print("Perguntas de teste:", len(teste))

exemplo = validacao[0]
print("\nPergunta:", exemplo["Question"])
print("Resposta correta:", exemplo["Answer"]["Value"])
print("Documentos associados:", len(exemplo.get("EntityPages", [])))

Perguntas de treino: 53988
Perguntas de validação: 7900
Perguntas de teste: 318

Pergunta: Where in England was Dame Judi Dench born?
Resposta correta: York
Documentos associados: 2


In [ ]:
exemplo = validacao[0]
documento = exemplo["EntityPages"][0]

caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]

print("Pergunta:", exemplo["Question"])
print("Resposta correta:", exemplo["Answer"]["Value"])
print("Ficheiro:", caminho)

with open(caminho, encoding="utf-8") as f:
    texto = f.read()

print("\nNúmero de palavras:", len(texto.split()))
print("\nInício do documento:\n")
print(texto[:2000])


Pergunta: Where in England was Dame Judi Dench born?
Resposta correta: York
Ficheiro: /content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa/evidence/wikipedia/England.txt

Número de palavras: 13835

Início do documento:

England is a country that is part of the United Kingdom.   It shares land borders with Scotland to the north and Wales to the west. The Irish Sea lies northwest of England and the Celtic Sea lies to the southwest. England is separated from continental Europe by the North Sea to the east and the English Channel to the south. The country covers much of the central and southern part of the island of Great Britain, which lies in the North Atlantic; and includes over 100 smaller islands such as the Isles of Scilly, and the Isle of Wight.

The area now called England was first inhabited by modern humans during the Upper Palaeolithic period, but takes its name from the Angles, one of the Germanic tribes who settled during the 5th and 6th centuries. England became a unifie

In [ ]:
def dividir_em_excertos(texto, tamanho=200, sobreposicao=50):
    if not 0 <= sobreposicao < tamanho:
        raise ValueError("A sobreposição deve estar entre 0 e tamanho - 1.")

    palavras = texto.split()
    excertos = []
    passo = tamanho - sobreposicao

    for inicio in range(0, len(palavras), passo):
        excertos.append(" ".join(palavras[inicio:inicio + tamanho]))

        if inicio + tamanho >= len(palavras):
            break

    return excertos

excertos = dividir_em_excertos(texto)

print("Total de excertos:", len(excertos))
print("Palavras no primeiro excerto:", len(excertos[0].split()))
print("\nPrimeiro excerto:\n")
print(excertos[0])

Total de excertos: 92
Palavras no primeiro excerto: 200

Primeiro excerto:

England is a country that is part of the United Kingdom. It shares land borders with Scotland to the north and Wales to the west. The Irish Sea lies northwest of England and the Celtic Sea lies to the southwest. England is separated from continental Europe by the North Sea to the east and the English Channel to the south. The country covers much of the central and southern part of the island of Great Britain, which lies in the North Atlantic; and includes over 100 smaller islands such as the Isles of Scilly, and the Isle of Wight. The area now called England was first inhabited by modern humans during the Upper Palaeolithic period, but takes its name from the Angles, one of the Germanic tribes who settled during the 5th and 6th centuries. England became a unified state in the 10th century, and since the Age of Discovery, which began during the 15th century, has had a significant cultural and legal impact on the

In [ ]:
excertos_exemplo = []

for documento in validacao[0]["EntityPages"]:
    caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]

    with open(caminho, encoding="utf-8") as f:
        conteudo = f.read()

    partes = dividir_em_excertos(conteudo)

    for numero, parte in enumerate(partes):
        excertos_exemplo.append({
            "ficheiro": documento["Filename"],
            "numero": numero,
            "texto": parte
        })

    print(f"{documento['Filename']}: {len(partes)} excertos")

print("\nTotal de excertos para esta pergunta:", len(excertos_exemplo))

England.txt: 92 excertos
Judi_Dench.txt: 39 excertos

Total de excertos para esta pergunta: 131


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pergunta = validacao[0]["Question"]
textos = [excerto["texto"] for excerto in excertos_exemplo]

vetorizador = TfidfVectorizer(stop_words="english")
matriz = vetorizador.fit_transform(textos)
vetor_pergunta = vetorizador.transform([pergunta])

pontuacoes = cosine_similarity(vetor_pergunta, matriz).ravel()
melhores = pontuacoes.argsort()[::-1][:5]

print("Pergunta:", pergunta)

for posicao, indice in enumerate(melhores, start=1):
    excerto = excertos_exemplo[indice]

    print(f"\n--- Resultado {posicao} ---")
    print("Ficheiro:", excerto["ficheiro"])
    print("Pontuação:", round(float(pontuacoes[indice]), 3))
    print(excerto["texto"])

Pergunta: Where in England was Dame Judi Dench born?

--- Resultado 1 ---
Ficheiro: Judi_Dench.txt
Pontuação: 0.246
Screen Actors Guild Awards, two Golden Globes, an Academy Award, and a Tony Award. She has also received the BAFTA Fellowship (2001) and the Special Olivier Award (2004). In June 2011, she received a fellowship from the British Film Institute (BFI). Dench is also a Fellow of the Royal Society of Arts (FRSA). Early life Dench was born in Heworth, North Riding of Yorkshire. Her mother, Eleanora Olive (née Jones), was born in Dublin. Her father, Reginald Arthur Dench, a doctor, was born in Dorset, and later moved to Dublin, where he was raised. He met Dench's mother while he was studying medicine at Trinity College, Dublin. Dench attended The Mount School, a Quaker independent secondary school in York, and became a Quaker. Her brothers, one of whom was actor Jeffery Dench, were born in Tyldesley, Lancashire. Her niece, Emma Dench, is a Roman historian and professor previousl

In [ ]:
from pathlib import Path
import subprocess
import sys
import importlib

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
REPO_DIR = Path("/content/triviaqa_official")

# Descarregar o repositório completo, incluindo os módulos auxiliares.
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/mandarjoshi90/triviaqa.git",
            str(REPO_DIR)
        ],
        check=True
    )

# Permitir que o Python encontre os módulos do repositório.
sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()

from evaluation import triviaqa_evaluation as scorer

print("Avaliador oficial carregado.")
print("Exemplo de normalização:", scorer.normalize_answer("The York!"))
print("Aliases disponíveis:", validacao[0]["Answer"]["NormalizedAliases"])

Avaliador oficial carregado.
Exemplo de normalização: york
Aliases disponíveis: ['york yorkshire', 'eoferwic', 'park grove primary school', 'park grove school', 'weather in york', 'park grove 1895', 'eoforwic', 'county borough of york', 'york uk', 'un locode gbyrk', 'city of york', 'york england', 'york ua', 'york ham', 'york', 'yorkish', 'yoisk', 'york north yorkshire']


In [ ]:
aliases = {
    scorer.normalize_answer(alias)
    for alias in validacao[0]["Answer"]["NormalizedAliases"]
}
aliases.discard("")

def contem_resposta(texto, aliases):
    texto_normalizado = " " + scorer.normalize_answer(texto) + " "
    return any(
        " " + alias + " " in texto_normalizado
        for alias in aliases
    )

# Ordenar todos os excertos pela pontuação da pesquisa TF-IDF.
ranking = pontuacoes.argsort()[::-1]

for k in [1, 5, 10]:
    encontrou = any(
        contem_resposta(excertos_exemplo[int(i)]["texto"], aliases)
        for i in ranking[:k]
    )

    print(f"Acerto de recuperação @{k}: {'SIM' if encontrou else 'NÃO'}")

Acerto de recuperação @1: SIM
Acerto de recuperação @5: SIM
Acerto de recuperação @10: SIM


In [ ]:
import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
pasta_divisoes = PROJECT_DIR / "data" / "splits"
pasta_divisoes.mkdir(parents=True, exist_ok=True)

conjuntos = {
    "treino": treino,
    "validacao": validacao,
    "teste": teste
}

# Confirmar que nenhuma pergunta aparece em conjuntos diferentes.
ids = {
    nome: {p["QuestionId"] for p in perguntas}
    for nome, perguntas in conjuntos.items()
}

assert ids["treino"].isdisjoint(ids["validacao"])
assert ids["treino"].isdisjoint(ids["teste"])
assert ids["validacao"].isdisjoint(ids["teste"])

for nome, perguntas in conjuntos.items():
    destino = pasta_divisoes / f"{nome}.json"

    with open(destino, "w", encoding="utf-8") as f:
        json.dump(
            perguntas, f,
            ensure_ascii=False
        )

    print(f"{nome}: {len(perguntas)} perguntas guardadas")

print("\nDivisões guardadas e sem perguntas em comum.")

treino: 53988 perguntas guardadas
validacao: 7900 perguntas guardadas
teste: 318 perguntas guardadas

Divisões guardadas e sem perguntas em comum.


In [ ]:
def pesquisar_excertos(pergunta, k=10):
    candidatos = []

    # Evitar ler documentos repetidos na mesma pergunta.
    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro

        with open(caminho, encoding="utf-8") as f:
            texto = f.read()

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            candidatos.append({
                "ficheiro": ficheiro,
                "numero": numero,
                "texto": excerto
            })

    if not candidatos:
        return []

    vetorizador = TfidfVectorizer(stop_words="english")
    matriz = vetorizador.fit_transform(
        [item["texto"] for item in candidatos]
    )
    consulta = vetorizador.transform([pergunta["Question"]])

    scores = cosine_similarity(consulta, matriz).ravel()
    indices = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in indices
    ]

# Testar noutra pergunta de validação.
resultados = pesquisar_excertos(validacao[1])

print("Pergunta:", validacao[1]["Question"])
print("Excertos recuperados:", len(resultados))

if resultados:
    print("\nPrimeiro resultado:", resultados[0]["ficheiro"])
    print(resultados[0]["texto"])

Pergunta: From which country did Angola achieve independence in 1975?
Excertos recuperados: 10

Primeiro resultado: Angola.txt
lasted several decades (with some interludes). It claimed millions of lives and produced many refugees; it didn't end until 2002. Following negotiations held in Portugal, itself experiencing severe social and political turmoil and uncertainty due to the April 1974 revolution, Angola's three main guerrilla groups agreed to establish a transitional government in January 1975. Within two months, however, the FNLA, MPLA and UNITA had started fighting each other and the country began splitting into zones controlled by rival armed political groups. The MPLA gained control of the capital Luanda and much of the rest of the country. With the support of the United States, Zaïre and South Africa intervened militarily in favour of the FNLA and UNITA with the intention of taking Luanda before the declaration of independence. In response, Cuba intervened in favor of the MPLA

In [ ]:
from tqdm.auto import tqdm
import json

avaliacao_inicial = []

for pergunta in tqdm(validacao[:20], desc="A avaliar"):
    recuperados = pesquisar_excertos(pergunta, k=10)

    aliases = {
        scorer.normalize_answer(alias)
        for alias in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    registo = {
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resultados": recuperados
    }

    for k in [1, 5, 10]:
        registo[f"acerto@{k}"] = any(
            contem_resposta(item["texto"], aliases)
            for item in recuperados[:k]
        )

    avaliacao_inicial.append(registo)

for k in [1, 5, 10]:
    acertos = sum(r[f"acerto@{k}"] for r in avaliacao_inicial)
    total = len(avaliacao_inicial)
    print(f"Recall@{k}: {acertos / total:.1%} ({acertos}/{total})")

destino = PROJECT_DIR / "results"
destino.mkdir(parents=True, exist_ok=True)

with open(destino / "tfidf_validacao_20.json", "w", encoding="utf-8") as f:
    json.dump(avaliacao_inicial, f, ensure_ascii=False, indent=2)

print("\nResultados guardados.")

A avaliar:   0%|          | 0/20 [00:00<?, ?it/s]

Recall@1: 60.0% (12/20)
Recall@5: 80.0% (16/20)
Recall@10: 90.0% (18/20)

Resultados guardados.


In [ ]:
perguntas_por_id = {
    p["QuestionId"]: p for p in validacao[:20]
}

for resultado in avaliacao_inicial:
    if resultado["acerto@10"]:
        continue

    pergunta = perguntas_por_id[resultado["id"]]

    print("=" * 70)
    print("Pergunta:", pergunta["Question"])
    print("Resposta correta:", pergunta["Answer"]["Value"])
    print("Aliases:", pergunta["Answer"]["NormalizedAliases"])

    # Verificar se a resposta existe em algum excerto dos documentos.
    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    todos = pesquisar_excertos(pergunta, k=100000)
    posicao = next(
        (
            i for i, item in enumerate(todos, start=1)
            if contem_resposta(item["texto"], aliases)
        ),
        None
    )

    print("Primeira posição com a resposta:", posicao)

    if resultado["resultados"]:
        print("\nPrimeiro excerto recuperado:")
        print(resultado["resultados"][0]["texto"])

    print()

Pergunta: Which was the first European country to abolish capital punishment?
Resposta correta: Norway
Aliases: ['norwegen', 'kongeriket norge', 'norway', 'republic of norway', 'noorwegen', 'norvege', 'mainland norway', 'kingdom of norway', 'sport in norway', 'noreg', 'noruega', 'norwegia', 'noregur', 'royal kingdom of norway', 'name of norway', 'kongeriket noreg', 'norwegian kingdom', 'etymology of norway', 'norvège', 'iso 3166 1 no', 'norwegian state']
Primeira posição com a resposta: 23

Primeiro excerto recuperado:
to life is the most important, and capital punishment violates it without necessity and inflicts to the condemned a psychological torture. Human rights activists oppose the death penalty, calling it "cruel, inhuman, and degrading punishment". Amnesty International considers it to be "the ultimate, irreversible denial of Human Rights". See also: Capital punishment abolition chronology. Contemporary use Capital punishment by country Most countries including almost all Firs

In [ ]:
diagnostico = []

for resultado in avaliacao_inicial:
    if resultado["acerto@10"]:
        continue

    pergunta = perguntas_por_id[resultado["id"]]

    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    todos = pesquisar_excertos(pergunta, k=100000)

    correspondencia = next(
        (
            (posicao, item)
            for posicao, item in enumerate(todos, start=1)
            if contem_resposta(item["texto"], aliases)
        ),
        None
    )

    diagnostico.append({
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resposta": pergunta["Answer"]["Value"],
        "primeira_posicao_com_alias": (
            correspondencia[0] if correspondencia else None
        ),
        "excerto_com_alias": (
            correspondencia[1] if correspondencia else None
        ),
        "primeiro_resultado": (
            resultado["resultados"][0]
            if resultado["resultados"] else None
        )
    })

destino = PROJECT_DIR / "results" / "diagnostico_tfidf_20.json"

with open(destino, "w", encoding="utf-8") as f:
    json.dump(diagnostico, f, ensure_ascii=False, indent=2)

print("Diagnóstico guardado.")

for caso in diagnostico:
    if caso["excerto_com_alias"]:
        print("\nPergunta:", caso["pergunta"])
        print("Excerto com alias:")
        print(caso["excerto_com_alias"]["texto"])

Diagnóstico guardado.

Pergunta: Which was the first European country to abolish capital punishment?
Excerto com alias:
others actively use it today. The death penalty there remains a contentious issue which is hotly debated. In abolitionist countries, the debate is sometimes revived by particularly brutal murders though few countries have brought it back after abolishing it. However, a spike in serious, violent crimes, such as murders or terrorist attacks, has prompted some countries (such as Sri Lanka and Jamaica) to effectively end the moratorium on the death penalty. In retention countries, the debate is sometimes revived when a miscarriage of justice has occurred though this tends to cause legislative efforts to improve the judicial process rather than to abolish the death penalty. Modern-day public opinion The public opinion on the death penalty varies considerably by country and by the crime in question. Countries where a majority of people are against execution include New Zeal

In [ ]:
avaliacao_200 = []

for pergunta in tqdm(validacao[:200], desc="A avaliar 200 perguntas"):
    recuperados = pesquisar_excertos(pergunta, k=10)

    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    registo = {
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resultados": recuperados
    }

    for k in [1, 5, 10]:
        registo[f"acerto@{k}"] = any(
            contem_resposta(item["texto"], aliases)
            for item in recuperados[:k]
        )

    avaliacao_200.append(registo)

    # Guardar o progresso a cada 20 perguntas.
    if len(avaliacao_200) % 20 == 0:
        destino = PROJECT_DIR / "results" / "tfidf_validacao_200.json"
        with open(destino, "w", encoding="utf-8") as f:
            json.dump(avaliacao_200, f, ensure_ascii=False, indent=2)

for k in [1, 5, 10]:
    acertos = sum(r[f"acerto@{k}"] for r in avaliacao_200)
    print(f"Recall@{k}: {acertos / len(avaliacao_200):.1%}")

print("\nResultados guardados.")

A avaliar 200 perguntas:   0%|          | 0/200 [00:00<?, ?it/s]

Recall@1: 58.0%
Recall@5: 83.0%
Recall@10: 87.0%

Resultados guardados.


In [ ]:
%pip -q install rank-bm25

In [ ]:
import re
from rank_bm25 import BM25Okapi

def tokenizar_bm25(texto):
    return re.findall(r"\w+", texto.lower())

def pesquisar_bm25(pergunta, k=10):
    candidatos = []

    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro

        with open(caminho, encoding="utf-8") as f:
            texto = f.read()

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            if tokenizar_bm25(excerto):
                candidatos.append({
                    "ficheiro": ficheiro,
                    "numero": numero,
                    "texto": excerto
                })

    if not candidatos:
        return []

    indice = BM25Okapi([
        tokenizar_bm25(item["texto"])
        for item in candidatos
    ])

    scores = indice.get_scores(tokenizar_bm25(pergunta["Question"]))
    melhores = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in melhores
    ]

avaliacao_bm25_200 = []

for pergunta in tqdm(validacao[:200], desc="A avaliar BM25"):
    recuperados = pesquisar_bm25(pergunta)

    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    registo = {
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resultados": recuperados
    }

    for k in [1, 5, 10]:
        registo[f"acerto@{k}"] = any(
            contem_resposta(item["texto"], aliases)
            for item in recuperados[:k]
        )

    avaliacao_bm25_200.append(registo)

    if len(avaliacao_bm25_200) % 20 == 0:
        destino = PROJECT_DIR / "results" / "bm25_validacao_200.json"
        with open(destino, "w", encoding="utf-8") as f:
            json.dump(
                avaliacao_bm25_200, f,
                ensure_ascii=False, indent=2
            )

print("\nComparação nas mesmas 200 perguntas:")
for k in [1, 5, 10]:
    tfidf = sum(r[f"acerto@{k}"] for r in avaliacao_200) / len(avaliacao_200)
    bm25 = sum(r[f"acerto@{k}"] for r in avaliacao_bm25_200) / len(avaliacao_bm25_200)

    print(f"Recall@{k}: TF-IDF = {tfidf:.1%} | BM25 = {bm25:.1%}")

print("\nResultados guardados.")

A avaliar BM25:   0%|          | 0/200 [00:00<?, ?it/s]


Comparação nas mesmas 200 perguntas:
Recall@1: TF-IDF = 58.0% | BM25 = 53.5%
Recall@5: TF-IDF = 83.0% | BM25 = 82.5%
Recall@10: TF-IDF = 87.0% | BM25 = 88.5%

Resultados guardados.


In [ ]:
import importlib.metadata
import subprocess
import json

configuracao = {
    "dataset": "TriviaQA RC — Wikipedia",
    "validacao": "Primeiras 7900 perguntas de wikipedia-train.json",
    "amostra_experimental": "Primeiras 200 perguntas da validação",
    "ambito_pesquisa": "Documentos associados a cada pergunta",
    "tamanho_excerto_palavras": 200,
    "sobreposicao_palavras": 50,
    "tfidf": {
        "stop_words": "english",
        "restantes_parametros": "Defaults do scikit-learn"
    },
    "bm25": {
        "tokenizacao": r"\w+ em minúsculas",
        "k1": 1.5,
        "b": 0.75,
        "epsilon": 0.25
    },
    "metrica": "Presença de alias normalizado com limites de palavras",
    "versoes": {
        pacote: importlib.metadata.version(pacote)
        for pacote in ["scikit-learn", "rank-bm25", "numpy"]
    },
    "commit_avaliador_oficial": subprocess.check_output(
        ["git", "-C", "/content/triviaqa_official", "rev-parse", "HEAD"],
        text=True
    ).strip()
}

destino = PROJECT_DIR / "results" / "configuracao_pesquisa_200.json"

with open(destino, "w", encoding="utf-8") as f:
    json.dump(configuracao, f, ensure_ascii=False, indent=2)

print("Configuração guardada.")

Configuração guardada.


In [ ]:
import torch

print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Modelo da GPU:", torch.cuda.get_device_name(0))
else:
    print("A sessão está a usar CPU.")

GPU disponível: True
Modelo da GPU: Tesla T4


In [ ]:
%pip -q install transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("Ativa a GPU no Colab antes de executar esta célula.")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

modelo.eval()

print("Modelo carregado:", MODEL_ID)
print("Dispositivo:", modelo.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo carregado: Qwen/Qwen2.5-1.5B-Instruct
Dispositivo: cuda:0


In [ ]:
from huggingface_hub import model_info

MODEL_REVISION = model_info(MODEL_ID).sha

print("Modelo carregado:", MODEL_ID)
print("Dispositivo:", modelo.device)
print("Versão no Hugging Face:", MODEL_REVISION)

Modelo carregado: Qwen/Qwen2.5-1.5B-Instruct
Dispositivo: cuda:0
Versão no Hugging Face: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


In [ ]:
def gerar_resposta(pergunta, contexto=""):
    mensagens = [
        {
            "role": "system",
            "content": (
                "Answer the question with only a short answer in English. "
                "Use the context when it is helpful. "
                "If the context is empty, answer using your knowledge. "
                "Do not include explanations."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{contexto}\n\n"
                f"Question: {pergunta}\n\nAnswer:"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(modelo.device)

    with torch.inference_mode():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = saida[0, entradas["input_ids"].shape[1]:]

    return tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    ).strip()


pergunta_teste = "From which country did Angola achieve independence in 1975?"

print("Pergunta:", pergunta_teste)
print("Resposta sem documentos:", gerar_resposta(pergunta_teste))

Pergunta: From which country did Angola achieve independence in 1975?
Resposta sem documentos: Portugal


In [ ]:
pergunta = validacao[1]

recuperados = pesquisar_excertos(pergunta, k=5)

contexto = "\n\n".join(
    f"[Excerto {i} | {item['ficheiro']}]\n{item['texto']}"
    for i, item in enumerate(recuperados, start=1)
)

resposta_rag = gerar_resposta(
    pergunta["Question"],
    contexto=contexto
)

print("Pergunta:", pergunta["Question"])
print("Excertos fornecidos:", len(recuperados))
print("Resposta com RAG:", resposta_rag)
print("Resposta correta:", pergunta["Answer"]["Value"])

NameError: name 'validacao' is not defined

In [ ]:
import json
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
DATA_DIR = PROJECT_DIR / "data" / "triviaqa"

with open(PROJECT_DIR / "data/splits/validacao.json", encoding="utf-8") as f:
    validacao = json.load(f)

print("Perguntas de validação recuperadas:", len(validacao))
print("Função de pesquisa disponível:",
      callable(globals().get("pesquisar_excertos")))

Mounted at /content/drive
Perguntas de validação recuperadas: 7900
Função de pesquisa disponível: False


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def dividir_em_excertos(texto, tamanho=200, sobreposicao=50):
    if not 0 <= sobreposicao < tamanho:
        raise ValueError("Sobreposição inválida.")

    palavras = texto.split()
    excertos = []

    for inicio in range(0, len(palavras), tamanho - sobreposicao):
        excertos.append(" ".join(palavras[inicio:inicio + tamanho]))

        if inicio + tamanho >= len(palavras):
            break

    return excertos


def pesquisar_excertos(pergunta, k=10):
    candidatos = []

    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro

        with open(caminho, encoding="utf-8") as f:
            texto = f.read()

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            candidatos.append({
                "ficheiro": ficheiro,
                "numero": numero,
                "texto": excerto
            })

    if not candidatos:
        return []

    vetorizador = TfidfVectorizer(stop_words="english")
    matriz = vetorizador.fit_transform(
        [item["texto"] for item in candidatos]
    )
    consulta = vetorizador.transform([pergunta["Question"]])

    scores = cosine_similarity(consulta, matriz).ravel()
    indices = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in indices
    ]


pergunta = validacao[1]
recuperados = pesquisar_excertos(pergunta, k=5)

contexto = "\n\n".join(
    f"[Excerto {i} | {item['ficheiro']}]\n{item['texto']}"
    for i, item in enumerate(recuperados, start=1)
)

resposta_rag = gerar_resposta(pergunta["Question"], contexto=contexto)

print("Pergunta:", pergunta["Question"])
print("Excertos fornecidos:", len(recuperados))
print("Resposta com RAG:", resposta_rag)
print("Resposta correta:", pergunta["Answer"]["Value"])

OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa/evidence/wikipedia/Nation_state.txt'

In [ ]:
import shutil
import tarfile
from pathlib import Path, PurePosixPath

arquivo_drive = PROJECT_DIR / "data" / "triviaqa-rc.tar.gz"
arquivo_local = Path("/content/triviaqa-rc.tar.gz")
DATA_DIR = Path("/content/triviaqa_local")

if not arquivo_local.exists():
    print("A copiar o arquivo para o Colab...")
    temporario = Path("/content/triviaqa-rc.tar.gz.part")
    shutil.copyfile(arquivo_drive, temporario)
    temporario.replace(arquivo_local)

necessarios = {
    doc["Filename"] for doc in validacao[1]["EntityPages"]
}

print("A extrair os documentos da pergunta...")

with tarfile.open(arquivo_local, "r:gz") as tar:
    for membro in tar:
        partes = PurePosixPath(membro.name).parts

        if not membro.isfile():
            continue

        if "evidence" not in partes:
            continue

        relativo = PurePosixPath(*partes[partes.index("evidence"):])

        if str(relativo).startswith("evidence/wikipedia/"):
            nome = str(relativo.relative_to("evidence/wikipedia"))

            if nome in necessarios:
                destino = DATA_DIR / str(relativo)
                destino.parent.mkdir(parents=True, exist_ok=True)

                with tar.extractfile(membro) as origem:
                    with open(destino, "wb") as saida:
                        shutil.copyfileobj(origem, saida)

                print("Extraído:", nome)

print("Documentos locais preparados.")

A copiar o arquivo para o Colab...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa-rc.tar.gz'

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")

for pasta in [PROJECT_DIR, PROJECT_DIR / "data", Path("/content")]:
    print(f"\nConteúdo de {pasta}:")

    if not pasta.is_dir():
        print("Pasta não encontrada.")
        continue

    for item in pasta.iterdir():
        print("[PASTA]" if item.is_dir() else "[FICHEIRO]", item.name)


Conteúdo de /content/drive/MyDrive/Projeto_RAG_TriviaQA:
[PASTA] data
[PASTA] results
[PASTA] checkpoints
[FICHEIRO] triviaqa_evaluation.py
[PASTA] __pycache__

Conteúdo de /content/drive/MyDrive/Projeto_RAG_TriviaQA/data:
[PASTA] triviaqa
[PASTA] splits

Conteúdo de /content:
[PASTA] .config
[PASTA] drive
[PASTA] sample_data


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

DATA_DIR = Path(
    "/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa"
)

# Confirmar que todos os documentos desta pergunta podem ser lidos.
for documento in validacao[1]["EntityPages"]:
    caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]

    with open(caminho, encoding="utf-8") as f:
        conteudo = f.read()

    print(documento["Filename"], "— leitura OK:", len(conteudo), "caracteres")

Mounted at /content/drive


OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa/evidence/wikipedia/Nation_state.txt'

In [ ]:
from pathlib import Path
import urllib.request
import tarfile

arquivo_local = Path("/content/triviaqa-rc.tar.gz")
DATA_DIR = Path("/content/triviaqa_local")
marcador = DATA_DIR / ".extracao_concluida"

if not marcador.exists():
    if not arquivo_local.exists():
        print("A descarregar o TriviaQA para o Colab...", flush=True)

        temporario = Path("/content/triviaqa-rc.tar.gz.part")
        urllib.request.urlretrieve(
            "https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz",
            temporario
        )
        temporario.replace(arquivo_local)
        print("Download concluído.", flush=True)

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    print("A extrair os documentos localmente...", flush=True)
    with tarfile.open(arquivo_local, "r:gz") as tar:
        tar.extractall(path=DATA_DIR, filter="data")

    marcador.touch()
    print("Extração concluída.", flush=True)

for documento in validacao[1]["EntityPages"]:
    caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]
    with open(caminho, encoding="utf-8") as f:
        texto = f.read()

    print(documento["Filename"], "— leitura OK:", len(texto), "caracteres")

A descarregar o TriviaQA para o Colab...
Download concluído.
A extrair os documentos localmente...
Extração concluída.
Nation_state.txt — leitura OK: 37123 caracteres
Angola.txt — leitura OK: 52426 caracteres
Angolan_Civil_War.txt — leitura OK: 74820 caracteres


In [ ]:
import json
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
DATA_DIR = Path("/content/triviaqa_local")

ficheiro = PROJECT_DIR / "data" / "splits" / "validacao.json"
with ficheiro.open(encoding="utf-8") as f:
    validacao = json.load(f)

print("Perguntas recuperadas:", len(validacao))
print("Documentos locais:", (DATA_DIR / "evidence" / "wikipedia" / "Nation_state.txt").is_file())
print("Pesquisa pronta:", callable(globals().get("pesquisar_excertos")))
print("Gerador pronto:", callable(globals().get("gerar_resposta")))
print("Modelo carregado:", "modelo" in globals() and "tokenizer" in globals())

Mounted at /content/drive
Perguntas recuperadas: 7900
Documentos locais: False
Pesquisa pronta: False
Gerador pronto: False
Modelo carregado: False


In [ ]:
from pathlib import Path
import tarfile
import urllib.request

DATA_DIR = Path("/content/triviaqa_local")
arquivo = Path("/content/triviaqa-rc.tar.gz")
marcador = DATA_DIR / ".extracao_concluida"

if not arquivo.is_file() and not marcador.is_file():
    print("A descarregar o TriviaQA...", flush=True)
    temporario = arquivo.with_suffix(".gz.part")
    urllib.request.urlretrieve(
        "https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz",
        temporario
    )
    temporario.replace(arquivo)
    print("Download concluído.", flush=True)

if not marcador.is_file():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("A extrair os documentos...", flush=True)
    with tarfile.open(arquivo, "r:gz") as tar:
        tar.extractall(path=DATA_DIR, filter="data")
    marcador.touch()

for documento in validacao[1]["EntityPages"]:
    caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]
    texto = caminho.read_text(encoding="utf-8")
    print(f"{caminho.name} — leitura OK ({len(texto)} caracteres)")

A descarregar o TriviaQA...
Download concluído.
A extrair os documentos...
Nation_state.txt — leitura OK (37123 caracteres)
Angola.txt — leitura OK (52426 caracteres)
Angolan_Civil_War.txt — leitura OK (74820 caracteres)


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("Esta sessão do Colab está sem GPU.")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION
)

modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    device_map={"": 0}
)

modelo.eval()
print("Modelo carregado em", modelo.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado em cuda:0


In [3]:
import json
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

ficheiro = Path(
    "/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/splits/validacao.json"
)

with ficheiro.open(encoding="utf-8") as f:
    validacao = json.load(f)

print("Perguntas de validação carregadas:", len(validacao))

Mounted at /content/drive
Perguntas de validação carregadas: 7900


In [6]:
from pathlib import Path
import tarfile
import urllib.request

DATA_DIR = Path("/content/triviaqa_local")
arquivo = Path("/content/triviaqa-rc.tar.gz")

documentos = [
    DATA_DIR / "evidence" / "wikipedia" / doc["Filename"]
    for doc in validacao[1]["EntityPages"]
]

if not all(caminho.is_file() for caminho in documentos):
    if not arquivo.is_file():
        print("A descarregar o TriviaQA para o Colab...", flush=True)
        temporario = arquivo.with_suffix(".gz.part")
        urllib.request.urlretrieve(
            "https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz",
            temporario
        )
        temporario.replace(arquivo)

    print("A extrair o TriviaQA...", flush=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(arquivo, "r:gz") as tar:
        tar.extractall(path=DATA_DIR, filter="data")

for caminho in documentos:
    texto = caminho.read_text(encoding="utf-8")
    print(f"{caminho.name} — leitura OK ({len(texto)} caracteres)")

A descarregar o TriviaQA para o Colab...
A extrair o TriviaQA...
Nation_state.txt — leitura OK (37123 caracteres)
Angola.txt — leitura OK (52426 caracteres)
Angolan_Civil_War.txt — leitura OK (74820 caracteres)


In [7]:
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("/content/triviaqa_local")

def dividir_em_excertos(texto, tamanho=200, sobreposicao=50):
    palavras = texto.split()
    excertos = []

    for inicio in range(0, len(palavras), tamanho - sobreposicao):
        excertos.append(" ".join(palavras[inicio:inicio + tamanho]))
        if inicio + tamanho >= len(palavras):
            break

    return excertos


def pesquisar_excertos(pergunta, k=10):
    candidatos = []
    ficheiros = dict.fromkeys(doc["Filename"] for doc in pergunta["EntityPages"])

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro
        texto = caminho.read_text(encoding="utf-8")

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            candidatos.append({
                "ficheiro": ficheiro,
                "numero": numero,
                "texto": excerto
            })

    if not candidatos:
        return []

    vetorizador = TfidfVectorizer(stop_words="english")
    matriz = vetorizador.fit_transform([item["texto"] for item in candidatos])
    consulta = vetorizador.transform([pergunta["Question"]])

    scores = cosine_similarity(consulta, matriz).ravel()
    indices = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in indices
    ]


def gerar_resposta(pergunta, contexto=""):
    mensagens = [
        {
            "role": "system",
            "content": (
                "Answer the question with only a short answer in English. "
                "Use the context when it is helpful. "
                "If the context is empty, answer using your knowledge. "
                "Do not include explanations."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{contexto}\n\nQuestion: {pergunta}\n\nAnswer:"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(modelo.device)

    with torch.inference_mode():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = saida[0, entradas["input_ids"].shape[1]:]
    return tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()


pergunta = validacao[1]
recuperados = pesquisar_excertos(pergunta, k=5)
contexto = "\n\n".join(
    f"[Excerto {i} | {item['ficheiro']}]\n{item['texto']}"
    for i, item in enumerate(recuperados, start=1)
)

print("Pergunta:", pergunta["Question"])
print("Excertos recuperados:", len(recuperados))
print("Resposta sem documentos:", gerar_resposta(pergunta["Question"]))
print("Resposta com RAG:", gerar_resposta(pergunta["Question"], contexto))
print("Resposta de referência:", pergunta["Answer"]["Value"])

Pergunta: From which country did Angola achieve independence in 1975?
Excertos recuperados: 5
Resposta sem documentos: Portugal
Resposta com RAG: Portugal
Resposta de referência: Portugal


In [8]:
import re

pergunta = validacao[1]
referencia = pergunta["Answer"]["Value"]

# Procurar em todos os excertos dos documentos associados.
todos = pesquisar_excertos(pergunta, k=None)
padrao = re.compile(rf"\b{re.escape(referencia)}\b", re.IGNORECASE)

excerto_oracle = next(
    (item for item in todos if padrao.search(item["texto"])),
    None
)

if excerto_oracle is None:
    print("Nenhum excerto contém a resposta de referência.")
else:
    contexto_oracle = (
        f"[Excerto 1 | {excerto_oracle['ficheiro']}]\n"
        f"{excerto_oracle['texto']}"
    )
    print("Documento Oracle:", excerto_oracle["ficheiro"])
    print("Resposta Oracle:", gerar_resposta(pergunta["Question"], contexto_oracle))
    print("Resposta de referência:", referencia)

Documento Oracle: Angola.txt
Resposta Oracle: Portugal
Resposta de referência: Portugal


In [9]:
import json
import subprocess
import sys
from pathlib import Path
from tqdm.auto import tqdm

# Repor o avaliador oficial se a sessão do Colab foi reiniciada.
repo = Path("/content/triviaqa_official")
if not (repo / "evaluation" / "triviaqa_evaluation.py").is_file():
    subprocess.run(
        ["git", "clone", "https://github.com/mandarjoshi90/triviaqa.git", str(repo)],
        check=True
    )

sys.path.insert(0, str(repo))
from evaluation import triviaqa_evaluation as scorer


def formatar_contexto(excertos):
    return "\n\n".join(
        f"[Excerto {i} | {item['ficheiro']}]\n{item['texto']}"
        for i, item in enumerate(excertos, start=1)
    )


resultados_piloto = []

for pergunta in tqdm(validacao[:20], desc="A comparar as três condições"):
    enunciado = pergunta["Question"]
    resposta = pergunta["Answer"]

    aliases = {
        scorer.normalize_answer(alias)
        for alias in [resposta["Value"], *resposta["NormalizedAliases"]]
    }
    aliases.discard("")

    ranking = pesquisar_excertos(pergunta, k=None)
    excertos_oracle = []

    for item in ranking:
        texto = f" {scorer.normalize_answer(item['texto'])} "
        if any(f" {alias} " in texto for alias in aliases):
            excertos_oracle.append(item)

    previsoes = {
        "closed_book": gerar_resposta(enunciado),
        "rag_tfidf": gerar_resposta(enunciado, formatar_contexto(ranking[:5])),
        "oracle": (
            gerar_resposta(enunciado, formatar_contexto(excertos_oracle[:5]))
            if excertos_oracle else None
        )
    }

    resultados_piloto.append({
        "id": pergunta["QuestionId"],
        "pergunta": enunciado,
        "referencia": resposta["Value"],
        "aliases": sorted(aliases),
        "previsoes": previsoes
    })

# Comparar as três condições nas mesmas perguntas.
comparaveis = [
    linha for linha in resultados_piloto
    if linha["previsoes"]["oracle"] is not None
]

print(f"\nPerguntas com excerto Oracle: {len(comparaveis)}/{len(resultados_piloto)}")

if comparaveis:
    for condicao in ("closed_book", "rag_tfidf", "oracle"):
        em = []
        f1 = []

        for linha in comparaveis:
            previsao = linha["previsoes"][condicao]
            em.append(max(scorer.exact_match_score(previsao, a) for a in linha["aliases"]))
            f1.append(max(scorer.f1_score(previsao, a) for a in linha["aliases"]))

        print(f"{condicao}: EM = {sum(em)/len(em):.1%} | F1 = {sum(f1)/len(f1):.1%}")

destino = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA/results")
destino.mkdir(parents=True, exist_ok=True)
(destino / "geracao_validacao_20.json").write_text(
    json.dumps(resultados_piloto, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Resultados guardados no Drive.")

A comparar as três condições:   0%|          | 0/20 [00:00<?, ?it/s]


Perguntas com excerto Oracle: 19/20
closed_book: EM = 5.3% | F1 = 10.9%
rag_tfidf: EM = 42.1% | F1 = 54.2%
oracle: EM = 57.9% | F1 = 69.5%
Resultados guardados no Drive.


In [10]:
ficheiro_bm25 = destino / "bm25_validacao_200.json"
bm25_guardado = json.loads(ficheiro_bm25.read_text(encoding="utf-8"))
bm25_por_id = {linha["id"]: linha["resultados"] for linha in bm25_guardado}

piloto_com_bm25 = []

for linha in tqdm(resultados_piloto, desc="A gerar respostas com RAG BM25"):
    contexto = formatar_contexto(bm25_por_id[linha["id"]][:5])
    resposta_bm25 = gerar_resposta(linha["pergunta"], contexto)

    piloto_com_bm25.append({
        **linha,
        "previsoes": {**linha["previsoes"], "rag_bm25": resposta_bm25}
    })

comparaveis = [
    linha for linha in piloto_com_bm25
    if linha["previsoes"]["oracle"] is not None
]

print(f"\nPerguntas comparadas: {len(comparaveis)}")

for condicao in ("closed_book", "rag_tfidf", "rag_bm25", "oracle"):
    em, f1 = [], []

    for linha in comparaveis:
        previsao = linha["previsoes"][condicao]
        em.append(max(scorer.exact_match_score(previsao, a) for a in linha["aliases"]))
        f1.append(max(scorer.f1_score(previsao, a) for a in linha["aliases"]))

    print(f"{condicao}: EM = {sum(em)/len(em):.1%} | F1 = {sum(f1)/len(f1):.1%}")

(destino / "geracao_validacao_20_com_bm25.json").write_text(
    json.dumps(piloto_com_bm25, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Resultados guardados no Drive.")

A gerar respostas com RAG BM25:   0%|          | 0/20 [00:00<?, ?it/s]


Perguntas comparadas: 19
closed_book: EM = 5.3% | F1 = 10.9%
rag_tfidf: EM = 42.1% | F1 = 54.2%
rag_bm25: EM = 42.1% | F1 = 57.0%
oracle: EM = 57.9% | F1 = 69.5%
Resultados guardados no Drive.


In [12]:
import json
from pathlib import Path
from tqdm.auto import tqdm

destino = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA/results")
ficheiro_resultados = destino / "geracao_validacao_200.json"


def ler_json(ficheiro):
    return json.loads(ficheiro.read_text(encoding="utf-8"))


def guardar_progresso(resultados):
    temporario = ficheiro_resultados.with_suffix(".tmp")
    temporario.write_text(
        json.dumps(list(resultados.values()), ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    temporario.replace(ficheiro_resultados)


tfidf_por_id = {
    linha["id"]: linha["resultados"]
    for linha in ler_json(destino / "tfidf_validacao_200.json")
}
bm25_por_id = {
    linha["id"]: linha["resultados"]
    for linha in ler_json(destino / "bm25_validacao_200.json")
}

ficheiro_inicio = (
    ficheiro_resultados if ficheiro_resultados.is_file()
    else destino / "geracao_validacao_20_com_bm25.json"
)
resultados = {
    linha["id"]: linha for linha in ler_json(ficheiro_inicio)
}

amostra = validacao[:200]
novas = 0

for pergunta in tqdm(amostra, desc="A avaliar a geração"):
    pid = pergunta["QuestionId"]
    if pid in resultados:
        continue

    enunciado = pergunta["Question"]
    resposta = pergunta["Answer"]
    aliases = {
        scorer.normalize_answer(alias)
        for alias in [resposta["Value"], *resposta["NormalizedAliases"]]
    }
    aliases.discard("")

    excertos_tfidf = tfidf_por_id[pid][:5]
    excertos_bm25 = bm25_por_id[pid][:5]

    # O Oracle pode procurar um alias fora dos primeiros resultados.
    excertos_oracle = []
    for item in pesquisar_excertos(pergunta, k=None):
        texto = f" {scorer.normalize_answer(item['texto'])} "
        if any(f" {alias} " in texto for alias in aliases):
            excertos_oracle.append(item)
            if len(excertos_oracle) == 5:
                break

    resultados[pid] = {
        "id": pid,
        "pergunta": enunciado,
        "referencia": resposta["Value"],
        "aliases": sorted(aliases),
        "previsoes": {
            "closed_book": gerar_resposta(enunciado),
            "rag_tfidf": gerar_resposta(enunciado, formatar_contexto(excertos_tfidf)),
            "rag_bm25": gerar_resposta(enunciado, formatar_contexto(excertos_bm25)),
            "oracle": (
                gerar_resposta(enunciado, formatar_contexto(excertos_oracle))
                if excertos_oracle else None
            )
        }
    }

    novas += 1
    if novas % 10 == 0:
        guardar_progresso(resultados)

guardar_progresso(resultados)

linhas = [resultados[p["QuestionId"]] for p in amostra]
com_oracle = [linha for linha in linhas if linha["previsoes"]["oracle"] is not None]
metodos = ("closed_book", "rag_tfidf", "rag_bm25")

for titulo, grupo, condicoes in [
    ("Validação completa", linhas, metodos),
    ("Subconjunto com Oracle", com_oracle, (*metodos, "oracle"))
]:
    print(f"\n{titulo} ({len(grupo)} perguntas)")
    if not grupo:
        continue

    for condicao in condicoes:
        em, f1 = [], []
        for linha in grupo:
            previsao = linha["previsoes"][condicao]
            em.append(max(scorer.exact_match_score(previsao, a) for a in linha["aliases"]))
            f1.append(max(scorer.f1_score(previsao, a) for a in linha["aliases"]))

        print(f"{condicao}: EM = {sum(em)/len(em):.1%} | F1 = {sum(f1)/len(f1):.1%}")

print("\nResultados guardados no Drive.")

A avaliar a geração:   0%|          | 0/200 [00:00<?, ?it/s]


Validação completa (200 perguntas)
closed_book: EM = 24.5% | F1 = 30.8%
rag_tfidf: EM = 54.5% | F1 = 63.2%
rag_bm25: EM = 55.0% | F1 = 62.6%

Subconjunto com Oracle (196 perguntas)
closed_book: EM = 25.0% | F1 = 31.5%
rag_tfidf: EM = 55.6% | F1 = 64.4%
rag_bm25: EM = 56.1% | F1 = 63.8%
oracle: EM = 61.2% | F1 = 69.7%

Resultados guardados no Drive.


In [14]:
%pip -q install rank-bm25

import re
from pathlib import Path
from rank_bm25 import BM25Okapi

DATA_DIR = Path("/content/triviaqa_local")

def tokenizar_bm25(texto):
    return re.findall(r"\w+", texto.lower())

def pesquisar_bm25(pergunta, k=10):
    candidatos = []
    tokens_candidatos = []
    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro
        texto = caminho.read_text(encoding="utf-8")

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            tokens = tokenizar_bm25(excerto)
            if tokens:
                candidatos.append({
                    "ficheiro": ficheiro,
                    "numero": numero,
                    "texto": excerto
                })
                tokens_candidatos.append(tokens)

    if not candidatos:
        return []

    indice = BM25Okapi(tokens_candidatos, k1=1.5, b=0.75, epsilon=0.25)
    scores = indice.get_scores(tokenizar_bm25(pergunta["Question"]))
    melhores = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in melhores
    ]

print("BM25 pronto.")

BM25 pronto.


In [15]:
ficheiro_teste = destino.parent / "data" / "splits" / "teste.json"
teste = ler_json(ficheiro_teste)
assert len(teste) == 318

metodos_pesquisa = {
    "tfidf": pesquisar_excertos,
    "bm25": pesquisar_bm25,
}

for nome, pesquisar in metodos_pesquisa.items():
    registos = []

    for pergunta in tqdm(teste, desc=f"Teste: {nome.upper()}"):
        resposta = pergunta["Answer"]
        aliases = {
            scorer.normalize_answer(a)
            for a in [resposta["Value"], *resposta["NormalizedAliases"]]
        } - {""}

        ranking = pesquisar(pergunta, k=10)
        acertos = []
        for excerto in ranking:
            texto = f" {scorer.normalize_answer(excerto['texto'])} "
            acertos.append(any(f" {alias} " in texto for alias in aliases))

        registos.append({
            "id": pergunta["QuestionId"],
            "resultados": ranking,
            **{f"acerto@{k}": any(acertos[:k]) for k in (1, 5, 10)},
        })

    (destino / f"{nome}_teste_318.json").write_text(
        json.dumps(registos, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"\n{nome.upper()} — {len(registos)} perguntas")
    for k in (1, 5, 10):
        acertos = sum(r[f"acerto@{k}"] for r in registos)
        print(f"Recall@{k}: {acertos / len(registos):.1%} ({acertos}/{len(registos)})")

Teste: TFIDF:   0%|          | 0/318 [00:00<?, ?it/s]


TFIDF — 318 perguntas
Recall@1: 65.1% (207/318)
Recall@5: 91.8% (292/318)
Recall@10: 96.9% (308/318)


Teste: BM25:   0%|          | 0/318 [00:00<?, ?it/s]


BM25 — 318 perguntas
Recall@1: 71.7% (228/318)
Recall@5: 92.5% (294/318)
Recall@10: 96.5% (307/318)


In [16]:
ficheiro_final = destino / "geracao_teste_318.json"

rankings_teste = {
    metodo: {
        linha["id"]: linha["resultados"]
        for linha in ler_json(destino / f"{metodo}_teste_318.json")
    }
    for metodo in ("tfidf", "bm25")
}

resultados_teste = {}
if ficheiro_final.is_file():
    resultados_teste = {
        linha["id"]: linha for linha in ler_json(ficheiro_final)
    }


def guardar_teste():
    temporario = ficheiro_final.with_suffix(".tmp")
    temporario.write_text(
        json.dumps(list(resultados_teste.values()), ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    temporario.replace(ficheiro_final)


pendentes = [p for p in teste if p["QuestionId"] not in resultados_teste]

for numero, pergunta in enumerate(tqdm(pendentes, desc="A gerar respostas"), start=1):
    pid = pergunta["QuestionId"]
    enunciado = pergunta["Question"]
    resposta = pergunta["Answer"]
    aliases = {
        scorer.normalize_answer(a)
        for a in [resposta["Value"], *resposta["NormalizedAliases"]]
    } - {""}

    excertos_oracle = []
    for item in pesquisar_excertos(pergunta, k=None):
        texto = f" {scorer.normalize_answer(item['texto'])} "
        if any(f" {alias} " in texto for alias in aliases):
            excertos_oracle.append(item)
            if len(excertos_oracle) == 5:
                break

    previsoes = {"closed_book": gerar_resposta(enunciado)}

    for metodo, rankings in rankings_teste.items():
        contexto = formatar_contexto(rankings[pid][:5])
        previsoes[f"rag_{metodo}"] = gerar_resposta(enunciado, contexto)

    previsoes["oracle"] = (
        gerar_resposta(enunciado, formatar_contexto(excertos_oracle))
        if excertos_oracle else None
    )

    resultados_teste[pid] = {
        "id": pid,
        "pergunta": enunciado,
        "referencia": resposta["Value"],
        "aliases": sorted(aliases),
        "previsoes": previsoes
    }

    if numero % 10 == 0:
        guardar_teste()

guardar_teste()

linhas = [resultados_teste[p["QuestionId"]] for p in teste]
com_oracle = [r for r in linhas if r["previsoes"]["oracle"] is not None]
metodos = ("closed_book", "rag_tfidf", "rag_bm25")

for titulo, grupo, condicoes in [
    ("Teste completo", linhas, metodos),
    ("Subconjunto com Oracle", com_oracle, (*metodos, "oracle"))
]:
    print(f"\n{titulo} ({len(grupo)} perguntas)")
    if not grupo:
        continue

    for condicao in condicoes:
        em, f1 = [], []
        for linha in grupo:
            previsao = linha["previsoes"][condicao]
            aliases = linha["aliases"]
            em.append(max(scorer.exact_match_score(previsao, a) for a in aliases))
            f1.append(max(scorer.f1_score(previsao, a) for a in aliases))

        print(f"{condicao}: EM = {sum(em)/len(em):.1%} | F1 = {sum(f1)/len(f1):.1%}")

print("\nResultados guardados no Drive.")

A gerar respostas:   0%|          | 0/318 [00:00<?, ?it/s]


Teste completo (318 perguntas)
closed_book: EM = 32.4% | F1 = 38.1%
rag_tfidf: EM = 63.5% | F1 = 72.1%
rag_bm25: EM = 69.5% | F1 = 76.2%

Subconjunto com Oracle (317 perguntas)
closed_book: EM = 32.5% | F1 = 38.2%
rag_tfidf: EM = 63.7% | F1 = 72.4%
rag_bm25: EM = 69.7% | F1 = 76.5%
oracle: EM = 77.0% | F1 = 84.3%

Resultados guardados no Drive.


In [17]:
from collections import Counter

linhas = ler_json(destino / "geracao_teste_318.json")
assert len(linhas) == 318

for metodo in ("tfidf", "bm25"):
    contagens = Counter()
    exemplos = {}

    for linha in linhas:
        previsoes = linha["previsoes"]
        aliases = linha["aliases"]

        sem = any(
            scorer.exact_match_score(previsoes["closed_book"], alias)
            for alias in aliases
        )
        com = any(
            scorer.exact_match_score(previsoes[f"rag_{metodo}"], alias)
            for alias in aliases
        )

        contagens[sem, com] += 1
        exemplos.setdefault((sem, com), linha)

    print(f"\nRAG com {metodo.upper()} vs. sem documentos")
    for estados, descricao in [
        ((False, True), "Ajudou: passou a acertar"),
        ((True, False), "Prejudicou: passou a falhar"),
        ((True, True), "Manteve o acerto"),
        ((False, False), "Manteve a falha"),
    ]:
        print(f"{descricao}: {contagens[estados]}")

    for estados in [(False, True), (True, False)]:
        if estados in exemplos:
            linha = exemplos[estados]
            print("\nPergunta:", linha["pergunta"])
            print("Referência:", linha["referencia"])
            print("Sem documentos:", linha["previsoes"]["closed_book"])
            print("Com RAG:", linha["previsoes"][f"rag_{metodo}"])


RAG com TFIDF vs. sem documentos
Ajudou: passou a acertar: 121
Prejudicou: passou a falhar: 22
Manteve o acerto: 81
Manteve a falha: 94

Pergunta: In which country is the castle that gives the Habsburg dynasty its name?
Referência: Switzerland
Sem documentos: Austria-Hungary
Com RAG: Switzerland

Pergunta: What is the Milky Way?
Referência: The local galaxy
Sem documentos: our galaxy
Com RAG: barrier to the formation of stars

RAG com BM25 vs. sem documentos
Ajudou: passou a acertar: 137
Prejudicou: passou a falhar: 19
Manteve o acerto: 84
Manteve a falha: 78

Pergunta: Who first drew Mickey Mouse when ?Disney first supplied the voice?
Referência: Ub Iwerks
Sem documentos: 1928
Com RAG: Ub Iwerks

Pergunta: What is the Milky Way?
Referência: The local galaxy
Sem documentos: our galaxy
Com RAG: barred spiral galaxy


In [18]:
from collections import Counter

linhas = ler_json(destino / "geracao_teste_318.json")
assert len(linhas) == 318

for metodo in ("tfidf", "bm25"):
    recuperacao = {
        registo["id"]: registo["acerto@5"]
        for registo in ler_json(destino / f"{metodo}_teste_318.json")
    }
    contagens = Counter()

    for linha in linhas:
        tem_alias = recuperacao[linha["id"]]
        previsao = linha["previsoes"][f"rag_{metodo}"]
        acertou = any(
            scorer.exact_match_score(previsao, alias)
            for alias in linha["aliases"]
        )
        contagens[tem_alias, acertou] += 1

    print(f"\n{metodo.upper()}")
    for tem_alias, descricao in [
        (True, "Com alias nos cinco excertos"),
        (False, "Sem alias nos cinco excertos"),
    ]:
        acertos = contagens[tem_alias, True]
        falhas = contagens[tem_alias, False]
        total = acertos + falhas
        if total:
            print(f"{descricao}: {acertos}/{total} acertos ({acertos/total:.1%}), {falhas} falhas")


TFIDF
Com alias nos cinco excertos: 198/292 acertos (67.8%), 94 falhas
Sem alias nos cinco excertos: 4/26 acertos (15.4%), 22 falhas

BM25
Com alias nos cinco excertos: 216/294 acertos (73.5%), 78 falhas
Sem alias nos cinco excertos: 5/24 acertos (20.8%), 19 falhas


In [19]:
import gc
import torch
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("Esta sessão do Colab está sem GPU.")

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = model_info(MODEL_ID).sha

# Libertar a memória ocupada pelo modelo anterior.
if "modelo" in globals():
    del modelo
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION
)
modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    device_map={"": 0}
)
modelo.eval()

print("Modelo:", MODEL_ID)
print("Revisão:", MODEL_REVISION)
print("Dispositivo:", modelo.device)
print("Resposta de teste:", gerar_resposta(validacao[1]["Question"]))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo: Qwen/Qwen2.5-3B-Instruct
Revisão: aa8e72537993ba99e69dfaafa59ed015b17504d1
Dispositivo: cuda:0
Resposta de teste: Angola


In [20]:
import json
from pathlib import Path
from tqdm.auto import tqdm

assert MODEL_ID == "Qwen/Qwen2.5-3B-Instruct", "Carrega primeiro o Qwen 3B."

destino = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA/results")
ficheiro_final = destino / "geracao_teste_318_qwen3b.json"

def ler_json(caminho):
    return json.loads(caminho.read_text(encoding="utf-8"))

teste = ler_json(destino.parent / "data" / "splits" / "teste.json")
assert len(teste) == 318

rankings_teste = {
    metodo: {
        linha["id"]: linha["resultados"]
        for linha in ler_json(destino / f"{metodo}_teste_318.json")
    }
    for metodo in ("tfidf", "bm25")
}

resultados_3b = {}
if ficheiro_final.is_file():
    resultados_3b = {
        linha["id"]: linha for linha in ler_json(ficheiro_final)
    }

def guardar_3b():
    temporario = ficheiro_final.with_suffix(".tmp")
    temporario.write_text(
        json.dumps(list(resultados_3b.values()), ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    temporario.replace(ficheiro_final)

pendentes = [p for p in teste if p["QuestionId"] not in resultados_3b]

for numero, pergunta in enumerate(
    tqdm(pendentes, desc="A avaliar Qwen 3B"), start=1
):
    pid = pergunta["QuestionId"]
    enunciado = pergunta["Question"]
    resposta = pergunta["Answer"]
    aliases = {
        scorer.normalize_answer(alias)
        for alias in [resposta["Value"], *resposta["NormalizedAliases"]]
    } - {""}

    excertos_oracle = []
    for item in pesquisar_excertos(pergunta, k=None):
        texto = f" {scorer.normalize_answer(item['texto'])} "
        if any(f" {alias} " in texto for alias in aliases):
            excertos_oracle.append(item)
            if len(excertos_oracle) == 5:
                break

    previsoes = {"closed_book": gerar_resposta(enunciado)}

    for metodo, rankings in rankings_teste.items():
        contexto = formatar_contexto(rankings[pid][:5])
        previsoes[f"rag_{metodo}"] = gerar_resposta(enunciado, contexto)

    previsoes["oracle"] = (
        gerar_resposta(enunciado, formatar_contexto(excertos_oracle))
        if excertos_oracle else None
    )

    resultados_3b[pid] = {
        "id": pid,
        "pergunta": enunciado,
        "referencia": resposta["Value"],
        "aliases": sorted(aliases),
        "previsoes": previsoes,
        "modelo": MODEL_ID,
        "revisao": MODEL_REVISION,
    }

    if numero % 10 == 0:
        guardar_3b()

guardar_3b()

linhas = [resultados_3b[p["QuestionId"]] for p in teste]
com_oracle = [linha for linha in linhas if linha["previsoes"]["oracle"] is not None]
metodos = ("closed_book", "rag_tfidf", "rag_bm25")

for titulo, grupo, condicoes in [
    ("Teste completo", linhas, metodos),
    ("Subconjunto com Oracle", com_oracle, (*metodos, "oracle")),
]:
    print(f"\n{titulo} ({len(grupo)} perguntas)")
    if not grupo:
        continue

    for condicao in condicoes:
        em, f1 = [], []

        for linha in grupo:
            previsao = linha["previsoes"][condicao]
            aliases = linha["aliases"]
            em.append(max(scorer.exact_match_score(previsao, a) for a in aliases))
            f1.append(max(scorer.f1_score(previsao, a) for a in aliases))

        print(f"{condicao}: EM = {sum(em)/len(em):.1%} | F1 = {sum(f1)/len(f1):.1%}")

print("\nResultados do Qwen 3B guardados em", ficheiro_final.name)

A avaliar Qwen 3B:   0%|          | 0/318 [00:00<?, ?it/s]


Teste completo (318 perguntas)
closed_book: EM = 40.9% | F1 = 47.4%
rag_tfidf: EM = 64.5% | F1 = 72.6%
rag_bm25: EM = 65.4% | F1 = 73.7%

Subconjunto com Oracle (317 perguntas)
closed_book: EM = 41.0% | F1 = 47.5%
rag_tfidf: EM = 64.7% | F1 = 72.9%
rag_bm25: EM = 65.6% | F1 = 74.0%
oracle: EM = 73.2% | F1 = 81.6%

Resultados do Qwen 3B guardados em geracao_teste_318_qwen3b.json
